# 🌸 藤原紀香 AIボイス Google Colab T4 GPU 超高速推論サーバー

HomeVoiceHub (Galaxy Glasses / Nest Hub Max / 外出先スマホ対話) 向けの **リアルタイム音声推論 (0.3秒)** サーバーです。

### 🚀 使い方 (わずか 1 ステップ)
1. 上部メニューの **「ランタイム」 > 「ランタイムのタイプを変更」** で **「T4 GPU」** が選択されていることを確認します。
2. 下のセルの **「▶」再生ボタン** を 1 回クリックします。
3. 自動で Python 3.10 完全隔離環境 ➔ 自宅PCから公式モデル取得 ➔ GPU推論サーバー起動 ➔ トンネル開通 ➔ **自宅PCへの自動ドッキング** まで一気に完走します！

> **常駐について**: このセルが実行されている間、Colab GPU で 0.3 秒の超高速推論が提供され続けます。

In [ ]:
# ==============================================================================
# 🌸 [1クリック全自動] 藤原紀香 AIボイス T4 GPU 超高速推論サーバー
# ==============================================================================
# 根本方針:
# 1. Colab システム更新耐性: uv により Python 3.10 完全隔離環境を自動構築（Colab 3.13 を 100% 遮断）
# 2. Style-Bert-VITS2 安定版 2.7.0 固定取得 & 黄金依存関係 (PyTorch 2.2.2 CUDA 12.1 / Transformers 4.40)
# 3. 自宅 PC からのモデル自動取得 & T4 GPU ロード & Cloudflare Tunnel による自宅自動ドッキング
# ==============================================================================

import os, sys, shutil
from pathlib import Path

# --- 1. 高速パッケージマネージャ uv の配備 ---
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
if not os.path.exists("/usr/local/bin/uv") and not os.path.exists("/root/.cargo/bin/uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh

# --- 2. Python 3.10 完全隔離環境の構築 (Colab システム Python 3.13 を 100% 遮断) ---
VENV_DIR = "/content/norika_env"
PY_BIN = f"{VENV_DIR}/bin/python"
if not os.path.exists(PY_BIN):
    print("📦 Python 3.10 完全隔離環境を構築中（約15秒）...")
    !uv venv --python 3.10 {VENV_DIR}

# --- 3. Style-Bert-VITS2 安定版 (ver 2.7.0) の取得 ---
if not os.path.exists("/content/Style-Bert-VITS2"):
    print("📥 Style-Bert-VITS2 安定版 (ver 2.7.0) を取得中...")
    !git clone -b 2.7.0 --depth 1 https://github.com/litagin02/Style-Bert-VITS2.git /content/Style-Bert-VITS2

# --- 4. 黄金動作バージョン依存関係の高速インストール ---
print("⚡ 黄金動作バージョン（PyTorch 2.2.2 CUDA 12.1 / Transformers 4.40 / NumPy 1.26.4 / Setuptools <80）を同期中...")
!uv pip install --python {PY_BIN} "torch==2.2.2" "torchaudio==2.2.2" --index-url https://download.pytorch.org/whl/cu121 --no-progress
!CMAKE_POLICY_VERSION_MINIMUM=3.5 uv pip install --python {PY_BIN} -r /content/Style-Bert-VITS2/requirements-colab.txt --no-progress
!uv pip install --python {PY_BIN} "transformers==4.40.2" "huggingface-hub==0.23.2" "numpy==1.26.4" "scipy==1.11.4" "setuptools<80" "soundfile" "requests" --no-progress

# --- 5. 日本語 BERT モデルの初期化 ---
%cd /content/Style-Bert-VITS2
!{PY_BIN} initialize.py --skip_default_models

# --- 6. 推論サーバーコードの配備 ＆ 起動 ---
server_code = '#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""\nGoogle Colab NVIDIA T4 GPU 音声推論サーバー (Style-Bert-VITS2 藤原紀香公式モデル)\n- 1セル実行で全自動セットアップ (GPU検証 -> モデル自動取得 -> サーバー起動 -> トンネル開通 -> 自宅PC自動登録)\n- 音声推論速度: 約 0.2〜0.3 秒 (手元 CPU の 25秒から 80倍高速化)\n"""\n\nimport os\nimport sys\nimport time\nimport json\nimport re\nimport shutil\nimport subprocess\nimport threading\nfrom pathlib import Path\nimport urllib.request\nimport urllib.parse\nfrom http.server import HTTPServer, BaseHTTPRequestHandler\nfrom socketserver import ThreadingMixIn\n\n# 自宅 PC の Cloudflare Tunnel URL (公開エンドポイント)\nHOME_TUNNEL_URL = "https://contracts-rather-kai-served.trycloudflare.com"\n\n# 作業ディレクトリ\nBASE_DIR = Path("/content/Style-Bert-VITS2") if Path("/content/Style-Bert-VITS2").exists() else (Path("/content/norika_gpu_server") if Path("/content").exists() else Path("./norika_gpu_server"))\nif Path("/content/Style-Bert-VITS2").exists():\n    sys.path.insert(0, "/content/Style-Bert-VITS2")\n\nMODEL_DIR = BASE_DIR / "models"\nMODEL_PATH = MODEL_DIR / "Norika_Official_e120_s480.safetensors"\nCONFIG_PATH = MODEL_DIR / "config.json"\nSTYLE_PATH = MODEL_DIR / "style_vectors.npy"\n\nPORT = 50050\n_tts_model = None\n\ndef check_gpu():\n    print("=======================================================", flush=True)\n    print("  [1/5] GPU 稼働状態の事前健全性チェック (Pre-flight Check)", flush=True)\n    print("=======================================================", flush=True)\n    try:\n        import torch\n        if not torch.cuda.is_available():\n            print("\\n❌ [ERROR] NVIDIA GPU が検出されませんでした！", flush=True)\n            print("👉 Colab メニューの「ランタイム」>「ランタイムのタイプを変更」から「T4 GPU」を選択してください。\\n", flush=True)\n            sys.exit(1)\n        device_name = torch.cuda.get_device_name(0)\n        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)\n        print(f"✅ GPU 検出成功: {device_name} (VRAM: {vram_gb:.1f} GB)", flush=True)\n        print(f"✅ PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}", flush=True)\n    except Exception as ex:\n        print(f"❌ GPU チェック失敗: {ex}", flush=True)\n        sys.exit(1)\n\ndef install_cloudflared():\n    print("\\n[2/5] Cloudflare Tunnel (cloudflared) の準備...", flush=True)\n    cf_bin = Path("/usr/local/bin/cloudflared")\n    if not cf_bin.exists():\n        print("  cloudflared をダウンロード中...", flush=True)\n        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"\n        try:\n            urllib.request.urlretrieve(url, str(cf_bin))\n            os.chmod(str(cf_bin), 0o755)\n            print("  ✅ cloudflared インストール完了", flush=True)\n        except Exception as ex:\n            print(f"  ❌ cloudflared ダウンロード失敗: {ex}", flush=True)\n    else:\n        print("  ✅ cloudflared 準備済み", flush=True)\n\ndef download_model_from_home():\n    print("\\n[3/5] 自宅 PC から藤原紀香 AIボイス公式モデルを自動取得...", flush=True)\n    MODEL_DIR.mkdir(parents=True, exist_ok=True)\n\n    files = [\n        ("config.json", CONFIG_PATH),\n        ("style_vectors.npy", STYLE_PATH),\n        ("Norika_Official_e120_s480.safetensors", MODEL_PATH)\n    ]\n\n    for fname, target_path in files:\n        if target_path.exists() and target_path.stat().st_size > 1000:\n            print(f"  ✅ 既存モデルあり: {fname} ({target_path.stat().st_size / 1024 / 1024:.1f} MB)", flush=True)\n            continue\n\n        remote_url = f"{HOME_TUNNEL_URL}/model/{fname}"\n        print(f"  ダウンロード開始: {remote_url} -> {target_path.name}...", flush=True)\n        try:\n            req = urllib.request.Request(remote_url, headers={"User-Agent": "ColabNorikaGpu/1.0"})\n            with urllib.request.urlopen(req, timeout=60) as resp, open(target_path, "wb") as out_f:\n                total_size = int(resp.headers.get("content-length", 0))\n                downloaded = 0\n                block_size = 1024 * 1024 # 1MB\n                while True:\n                    chunk = resp.read(block_size)\n                    if not chunk:\n                        break\n                    out_f.write(chunk)\n                    downloaded += len(chunk)\n                    if total_size > 0:\n                        pct = (downloaded / total_size) * 100\n                        print(f"\\r    進捗: {downloaded/1024/1024:.1f}MB / {total_size/1024/1024:.1f}MB ({pct:.1f}%)", end="", flush=True)\n            print(f"\\n  ✅ 取得完了: {fname} ({target_path.stat().st_size / 1024 / 1024:.1f} MB)", flush=True)\n        except Exception as ex:\n            print(f"\\n  ❌ ダウンロード失敗 ({fname}): {ex}", flush=True)\n            raise\n\ndef load_vits2_model():\n    global _tts_model\n    print("\\n[4/5] NVIDIA T4 GPU に Style-Bert-VITS2 モデルを一括ロード中...", flush=True)\n    t0 = time.time()\n    from style_bert_vits2.tts_model import TTSModel\n\n    _tts_model = TTSModel(\n        model_path=str(MODEL_PATH),\n        config_path=str(CONFIG_PATH),\n        style_vec_path=str(STYLE_PATH),\n        device="cuda"\n    )\n    load_time = time.time() - t0\n    print(f"✅ 藤原紀香公式モデル GPU ロード完了 ({load_time:.2f} 秒)！", flush=True)\n\n    # 初回ウォームアップ推論\n    print("  初回ウォームアップ推論中...", flush=True)\n    tw0 = time.time()\n    sr, _ = _tts_model.infer(text="こんにちは。ふじわらのりかです。")\n    print(f"  ✅ ウォームアップ完了 ({time.time() - tw0:.2f} 秒, サンプリングレート: {sr}Hz)", flush=True)\n\ndef normalize_text(text: str) -> str:\n    t = text.replace("！", "。").replace("!", "。").replace("？", "。").replace("?", "。")\n    t = t.replace("～", "ー").replace("〜", "ー").replace("・", "、")\n    t = re.sub(r"[^\\w\\s、。ー「」『』\\(\\)（）]", "", t)\n    t = re.sub(r"([。、])\\1+", r"\\1", t)\n    t = t.replace("藤原紀香", "ふじわらのりか")\n    if not t:\n        t = "はい。"\n    elif not t.endswith("。") and not t.endswith("、"):\n        t += "。"\n    return t\n\nclass ThreadingHTTPServer(ThreadingMixIn, HTTPServer):\n    daemon_threads = True\n\nclass NorikaGpuHandler(BaseHTTPRequestHandler):\n    def log_message(self, format, *args):\n        pass\n\n    def do_HEAD(self):\n        self.send_response(200)\n        self.send_header("Access-Control-Allow-Origin", "*")\n        self.end_headers()\n\n    def do_GET(self):\n        parsed = urllib.parse.urlparse(self.path)\n        if parsed.path == "/health":\n            self.send_response(200)\n            self.send_header("Content-Type", "application/json; charset=utf-8")\n            self.send_header("Access-Control-Allow-Origin", "*")\n            self.end_headers()\n            import torch\n            res = {\n                "status": "ok",\n                "engine": "Style-Bert-VITS2-GPU",\n                "device": "cuda",\n                "gpu": torch.cuda.get_device_name(0),\n                "model": MODEL_PATH.name\n            }\n            self.wfile.write(json.dumps(res, ensure_ascii=False).encode("utf-8"))\n            return\n\n        if parsed.path in ["/voice", "/synthesis"]:\n            qs = urllib.parse.parse_qs(parsed.query, encoding="utf-8")\n            text = qs.get("text", [""])[0]\n            if not text:\n                self.send_error(400, "text is required")\n                return\n\n            try:\n                norm_text = normalize_text(text)\n                t0 = time.time()\n                sr, audio_array = _tts_model.infer(text=norm_text)\n                infer_time = time.time() - t0\n\n                # WAV バイト列にエンコード\n                import io\n                import soundfile as sf\n                wav_io = io.BytesIO()\n                sf.write(wav_io, audio_array, sr, format="WAV", subtype="PCM_16")\n                wav_bytes = wav_io.getvalue()\n\n                print(f"[GPU INFER SUCCESS: {infer_time*1000:.1f}ms] {len(wav_bytes)/1024:.1f}KB for: 「{norm_text}」", flush=True)\n\n                self.send_response(200)\n                self.send_header("Content-Type", "audio/wav")\n                self.send_header("Content-Length", str(len(wav_bytes)))\n                self.send_header("Access-Control-Allow-Origin", "*")\n                self.send_header("X-Inference-Time-Ms", f"{infer_time*1000:.1f}")\n                self.end_headers()\n                self.wfile.write(wav_bytes)\n            except Exception as ex:\n                print(f"[GPU INFER ERROR] {ex}", flush=True)\n                self.send_error(500, str(ex))\n            return\n\n        self.send_error(404, "Not Found")\n\ndef start_server_and_tunnel():\n    print("\\n[5/5] HTTP 推論サーバー起動 ＆ Cloudflare Tunnel 自動接続...", flush=True)\n\n    # 1. ローカル HTTP サーバーをバックグラウンド起動\n    server = ThreadingHTTPServer(("127.0.0.1", PORT), NorikaGpuHandler)\n    server_thread = threading.Thread(target=server.serve_forever, daemon=True)\n    server_thread.start()\n    print(f"  ✅ ローカル推論サーバー稼働中: http://127.0.0.1:{PORT}", flush=True)\n\n    # 2. Cloudflare Tunnel 起動\n    cmd = ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"]\n    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n\n    tunnel_url = None\n    url_regex = re.compile(r"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com")\n\n    for _ in range(40):\n        line = proc.stdout.readline()\n        if not line:\n            time.sleep(0.5)\n            continue\n        match = url_regex.search(line)\n        if match:\n            tunnel_url = match.group(0)\n            break\n\n    if not tunnel_url:\n        print("❌ Cloudflare Tunnel URL の取得に失敗しました。再試行してください。", flush=True)\n        return\n\n    print(f"\\n=======================================================", flush=True)\n    print(f"  🎉 祝・開通！藤原紀香 AIボイス Google Colab GPU サーバー", flush=True)\n    print(f"  🔗 Colab Tunnel URL: {tunnel_url}", flush=True)\n    print(f"=======================================================", flush=True)\n\n    # 3. 自宅 PC の HomeVoiceHub へ自動登録\n    print(f"\\n  自宅 PC ({HOME_TUNNEL_URL}) へ Colab GPU を自動登録中...", flush=True)\n    try:\n        reg_url = f"{HOME_TUNNEL_URL}/api/colab/register"\n        req_data = json.dumps({"url": tunnel_url}).encode("utf-8")\n        req = urllib.request.Request(\n            reg_url,\n            data=req_data,\n            headers={"Content-Type": "application/json", "User-Agent": "ColabNorikaGpu/1.0"}\n        )\n        with urllib.request.urlopen(req, timeout=10) as resp:\n            resp_body = resp.read().decode("utf-8")\n            print(f"  ✅ 自宅 PC への自動登録完了！: {resp_body}", flush=True)\n    except Exception as ex:\n        print(f"  ⚠️ 自宅 PC への自動通知でエラー (手動でも可): {ex}", flush=True)\n\n    print("\\n🌸 すべての準備が完了しました！", flush=True)\n    print("👉 スマホから話しかけると、NVIDIA GPU により【0.3秒】で紀香ボイスが即座に返ってきます！", flush=True)\n    print("（このセルの実行を停止するまで、GPU推論サーバーは常駐し続けます）\\n", flush=True)\n\n    try:\n        while True:\n            time.sleep(1)\n    except KeyboardInterrupt:\n        print("\\nサーバーを停止しました。")\n        proc.terminate()\n        server.shutdown()\n\nif __name__ == "__main__":\n    check_gpu()\n    install_cloudflared()\n    download_model_from_home()\n    load_vits2_model()\n    start_server_and_tunnel()\n'
with open('/content/Style-Bert-VITS2/colab_norika_server.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

!{PY_BIN} /content/Style-Bert-VITS2/colab_norika_server.py
